<a href="https://colab.research.google.com/github/rasushi/AskDocs/blob/main/02_train_model2_unet_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

sushmithaveigas_fundus_avseg_path = kagglehub.dataset_download('sushmithaveigas/fundus-avseg')
sushmithaveigas_driave_path = kagglehub.dataset_download('sushmithaveigas/driave')
sushmithaveigas_dataprep_op_avseg_path = kagglehub.dataset_download('sushmithaveigas/dataprep-op-avseg')

print('Data source import complete.')


In [ ]:
!pip install segmentation-models-pytorch albumentations opencv-python-headless matplotlib --no-deps


In [ ]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import time
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
# ===== CONFIGURATION =====
CONFIG = {
    'model_name': 'unet_resnet50',  # Changed
    'architecture': 'UNet',
    'encoder': 'resnet50',  # Changed from 'resnet34'
    'encoder_weights': 'imagenet',
    'train_json': '/kaggle/input/dataprep-op-avseg/processed_data/train_files.json',
    'val_json': '/kaggle/input/dataprep-op-avseg/processed_data/val_files.json',
    'test_json': '/kaggle/input/dataprep-op-avseg/processed_data/test_files.json',
    'num_classes': 3,
    'in_channels': 3,
    'image_size': 512,
    'batch_size': 6,  # Reduced from 8 (ResNet50 is larger)
    'num_epochs': 100,
    'learning_rate': 0.0001,
    'weight_decay': 1e-5,
    'early_stopping_patience': 10,
    'output_dir': '/kaggle/working/model2_outputs',  # Changed
    'checkpoint_path': '/kaggle/working/model2_outputs/model2_best.pth',  # Changed
    'history_path': '/kaggle/working/model2_outputs/model2_history.csv',  # Changed
}
# Create output directory
os.makedirs(CONFIG['output_dir'], exist_ok=True)

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
class RetinalAVSegmentationDataset(Dataset):
    """
    Custom Dataset for Retinal Artery-Vein Segmentation
    Handles both DRIAVE and Fundus-AVSeg datasets
    """

    def __init__(self, json_path, transforms=None, image_size=512):
        """
        Args:
            json_path: Path to JSON file with image and mask paths
            transforms: Albumentations transforms
            image_size: Target image size (height, width)
        """
        with open(json_path, 'r') as f:
            self.data = json.load(f)

        self.transforms = transforms
        self.image_size = image_size

        print(f"Loaded {len(self.data)} samples from {json_path}")

    def __len__(self):
        return len(self.data)

    def load_image(self, image_path):
        """Load image (handles both .tif and .png)"""
        if image_path.endswith('.tif'):
            image = Image.open(image_path)
            image = np.array(image)
        else:
            image = cv2.imread(image_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return image

    def load_mask(self, mask_path):
        """Load and convert mask to 3-class format"""
        mask = cv2.imread(mask_path)
        mask_rgb = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)

        # Convert to 3-class mask
        class_mask = self.convert_to_class_mask(mask_rgb)
        return class_mask

    def convert_to_class_mask(self, color_mask):
        """
        Convert color-coded mask to class indices
        0 = Background (black + white + green)
        1 = Artery (red)
        2 = Vein (blue)
        """
        h, w, _ = color_mask.shape
        class_mask = np.zeros((h, w), dtype=np.int64)

        # Red (artery) - high R, low G, low B
        red_mask = (color_mask[:, :, 0] > 200) & \
                   (color_mask[:, :, 1] < 50) & \
                   (color_mask[:, :, 2] < 50)

        # Blue (vein) - low R, low G, high B
        blue_mask = (color_mask[:, :, 2] > 200) & \
                    (color_mask[:, :, 0] < 50) & \
                    (color_mask[:, :, 1] < 50)

        # Assign classes
        class_mask[red_mask] = 1
        class_mask[blue_mask] = 2

        return class_mask

    def __getitem__(self, idx):
        """Get a single sample"""
        sample = self.data[idx]

        # Load image and mask
        image = self.load_image(sample['image'])
        mask = self.load_mask(sample['mask'])

        # Apply transforms
        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']

        return {
            'image': image,
            'mask': mask,
            'id': sample['id'],
            'dataset': sample['dataset']
        }

# Test the dataset
print("\nTesting dataset loading...")
test_dataset = RetinalAVSegmentationDataset(
    CONFIG['train_json'],
    image_size=CONFIG['image_size']
)
print(f"✅ Dataset loaded: {len(test_dataset)} samples")

# Check a sample
sample = test_dataset[0]
print(f"\nSample check:")
print(f"  Image shape: {sample['image'].shape}")
print(f"  Mask shape: {sample['mask'].shape}")
print(f"  Mask unique values: {np.unique(sample['mask'])}")
print(f"  Dataset: {sample['dataset']}")

In [ ]:
def get_training_transforms(image_size=512):
    """
    Training augmentation - optimized for retinal vessel segmentation
    """
    return A.Compose([
        A.Resize(image_size, image_size),

        # Geometric transforms
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(
            shift_limit=0.05,
            scale_limit=0.1,
            rotate_limit=15,
            border_mode=cv2.BORDER_CONSTANT,
            p=0.5
        ),

        # Color/Intensity - important for retinal images
        A.RandomBrightnessContrast(
            brightness_limit=0.15,
            contrast_limit=0.15,
            p=0.5
        ),
        A.RandomGamma(gamma_limit=(85, 115), p=0.3),

        # CLAHE - improves vessel contrast
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),

        # Hue/Sat - gentle (preserve artery/vein color difference)
        A.HueSaturationValue(
            hue_shift_limit=5,
            sat_shift_limit=15,
            val_shift_limit=10,
            p=0.3
        ),

        # Blur/Noise - simulate image quality variation
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1),
            A.MedianBlur(blur_limit=3, p=1),
        ], p=0.2),

        A.GaussNoise(var_limit=(5.0, 15.0), p=0.2),

        # Normalize
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
        ToTensorV2(),
    ])


def get_validation_transforms(image_size=512):
    """
    Validation/test transforms - no augmentation
    """
    return A.Compose([
        A.Resize(image_size, image_size),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
        ToTensorV2(),
    ])


print("✅ Augmentation pipelines defined")


# Quick visualization
def visualize_augmentations(dataset, num_samples=2):
    """Show augmentation examples"""
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))

    train_transforms = get_training_transforms(CONFIG['image_size'])
    colors = {0: [0, 0, 0], 1: [255, 0, 0], 2: [0, 0, 255]}

    for i in range(num_samples):
        sample = dataset[i]
        image, mask = sample['image'], sample['mask']

        # Original
        axes[i, 0].imshow(image)
        axes[i, 0].set_title('Original')
        axes[i, 0].axis('off')

        # Augmented examples
        for j in range(1, 4):
            aug = train_transforms(image=image, mask=mask)
            aug_img = aug['image'].numpy().transpose(1, 2, 0)
            aug_img = aug_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            aug_img = np.clip(aug_img, 0, 1)

            axes[i, j].imshow(aug_img)
            axes[i, j].set_title(f'Aug {j}')
            axes[i, j].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], 'augmentation_examples.png'), dpi=150)
    plt.show()


visualize_augmentations(test_dataset, num_samples=2)

In [ ]:
# Create datasets with transforms
train_dataset = RetinalAVSegmentationDataset(
    CONFIG['train_json'],
    transforms=get_training_transforms(CONFIG['image_size']),
    image_size=CONFIG['image_size']
)

val_dataset = RetinalAVSegmentationDataset(
    CONFIG['val_json'],
    transforms=get_validation_transforms(CONFIG['image_size']),
    image_size=CONFIG['image_size']
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n{'='*60}")
print("DataLoaders Created")
print(f"{'='*60}")
print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Batch size:         {CONFIG['batch_size']}")
print(f"{'='*60}")

# Test dataloader
print("\nTesting dataloader...")
batch = next(iter(train_loader))
print(f"Batch images shape: {batch['image'].shape}")
print(f"Batch masks shape: {batch['mask'].shape}")
print(f"✅ DataLoaders working correctly")

In [ ]:
# Create model
model = smp.Unet(
    encoder_name=CONFIG['encoder'],
    encoder_weights=CONFIG['encoder_weights'],
    in_channels=CONFIG['in_channels'],
    classes=CONFIG['num_classes'],
)

model = model.to(device)

print(f"\n{'='*60}")
print("Model Architecture")
print(f"{'='*60}")
print(f"Architecture: {CONFIG['architecture']}")
print(f"Encoder: {CONFIG['encoder']}")
print(f"Encoder weights: {CONFIG['encoder_weights']}")
print(f"Input channels: {CONFIG['in_channels']}")
print(f"Output classes: {CONFIG['num_classes']}")
print(f"{'='*60}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Parameters:")
print(f"  Total: {total_params:,}")
print(f"  Trainable: {trainable_params:,}")
print(f"  Size: ~{total_params * 4 / 1e6:.2f} MB (fp32)")

In [ ]:
class DiceLoss(nn.Module):
    """Dice Loss for segmentation"""
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, predictions, targets):
        targets = targets.long()
        predictions = torch.softmax(predictions, dim=1)
        targets_one_hot = torch.nn.functional.one_hot(
            targets.long(),
            num_classes=predictions.shape[1]
        ).permute(0, 3, 1, 2).float()

        dice_scores = []
        for c in range(predictions.shape[1]):
            pred_c = predictions[:, c]
            target_c = targets_one_hot[:, c]

            intersection = (pred_c * target_c).sum()
            union = pred_c.sum() + target_c.sum()

            dice = (2. * intersection + self.smooth) / (union + self.smooth)
            dice_scores.append(dice)

        return 1 - torch.mean(torch.stack(dice_scores))


class TverskyLoss(nn.Module):
    """
    Tversky Loss - better for imbalanced segmentation
    Controls trade-off between false positives and false negatives
    alpha: weight for false positives
    beta: weight for false negatives
    When alpha=beta=0.5, it becomes Dice Loss
    """
    def __init__(self, alpha=0.3, beta=0.7, smooth=1.0):
        super(TverskyLoss, self).__init__()
        self.alpha = alpha  # Lower alpha = penalize FP less (good for small vessels)
        self.beta = beta    # Higher beta = penalize FN more (don't miss vessels)
        self.smooth = smooth

    def forward(self, predictions, targets):
        targets = targets.long()
        predictions = torch.softmax(predictions, dim=1)
        targets_one_hot = torch.nn.functional.one_hot(
            targets.long(),
            num_classes=predictions.shape[1]
        ).permute(0, 3, 1, 2).float()

        tversky_scores = []
        for c in range(predictions.shape[1]):
            pred_c = predictions[:, c]
            target_c = targets_one_hot[:, c]

            tp = (pred_c * target_c).sum()
            fp = ((1 - target_c) * pred_c).sum()
            fn = (target_c * (1 - pred_c)).sum()

            tversky = (tp + self.smooth) / (tp + self.alpha * fp + self.beta * fn + self.smooth)
            tversky_scores.append(tversky)

        return 1 - torch.mean(torch.stack(tversky_scores))


class FocalLoss(nn.Module):
    """
    Focal Loss - focuses on hard examples
    Good for class imbalance (background >> vessels)
    """
    def __init__(self, alpha=None, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha  # Class weights
        self.gamma = gamma  # Focusing parameter (higher = focus more on hard examples)

    def forward(self, predictions, targets):
        targets = targets.long()
        ce_loss = nn.CrossEntropyLoss(weight=self.alpha, reduction='none')(predictions, targets)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss


class CombinedLoss(nn.Module):
    """
    Combined Loss: Tversky + Focal
    - Tversky: handles class imbalance in segmentation
    - Focal: focuses on hard examples
    """
    def __init__(self, tversky_weight=0.7, focal_weight=0.3,
                 alpha=0.3, beta=0.7, gamma=2.0, class_weights=None):
        super(CombinedLoss, self).__init__()
        self.tversky_loss = TverskyLoss(alpha=alpha, beta=beta)
        self.focal_loss = FocalLoss(alpha=class_weights, gamma=gamma)
        self.tversky_weight = tversky_weight
        self.focal_weight = focal_weight

    def forward(self, predictions, targets):
        tversky = self.tversky_loss(predictions, targets)
        focal = self.focal_loss(predictions, targets)
        return self.tversky_weight * tversky + self.focal_weight * focal


def calculate_dice_score(predictions, targets, num_classes=3):
    targets = targets.long()
    """Calculate Dice score for evaluation"""
    predictions = torch.softmax(predictions, dim=1)
    predictions = torch.argmax(predictions, dim=1)

    dice_scores = {}

    for c in range(num_classes):
        pred_c = (predictions == c).float()
        target_c = (targets == c).float()

        intersection = (pred_c * target_c).sum()
        union = pred_c.sum() + target_c.sum()

        if union > 0:
            dice = (2. * intersection) / (union + 1e-8)
            dice_scores[f'class_{c}'] = dice.item()
        else:
            dice_scores[f'class_{c}'] = 0.0

    dice_scores['overall'] = np.mean([dice_scores['class_1'], dice_scores['class_2']])

    return dice_scores


# Calculate class weights from training data (optional but recommended)
def calculate_class_weights(dataloader, num_classes=3):
    """Calculate inverse frequency weights for class imbalance"""
    class_counts = torch.zeros(num_classes)

    print("Calculating class weights...")
    for batch in tqdm(dataloader):
        masks = batch['mask']
        for c in range(num_classes):
            class_counts[c] += (masks == c).sum().item()

    # Inverse frequency
    total_pixels = class_counts.sum()
    class_weights = total_pixels / (num_classes * class_counts)

    # Normalize
    class_weights = class_weights / class_weights.sum() * num_classes

    print(f"Class weights: {class_weights}")
    return class_weights


# Define loss and optimizer
print("Calculating class weights for Focal Loss...")
class_weights = calculate_class_weights(train_loader, num_classes=3)
class_weights = class_weights.to(device)

# Use Combined Loss (Tversky + Focal)
criterion = CombinedLoss(
    tversky_weight=0.7,
    focal_weight=0.3,
    alpha=0.3,        # FP weight (lower = less penalty for FP)
    beta=0.7,         # FN weight (higher = more penalty for FN)
    gamma=2.0,        # Focal focusing parameter
    class_weights=class_weights
)

optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay']
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
    verbose=True
)

print("✅ Optimizer and scheduler defined")

In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()

    running_loss = 0.0
    dice_scores_epoch = []

    pbar = tqdm(dataloader, desc='Training')

    for batch in pbar:
        images = batch['image'].to(device)
        masks = batch['mask'].to(device)

        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Calculate metrics
        with torch.no_grad():
            dice_scores = calculate_dice_score(outputs, masks, CONFIG['num_classes'])
            dice_scores_epoch.append(dice_scores['overall'])

        running_loss += loss.item()

        # Update progress bar
        pbar.set_postfix({
            'loss': f"{loss.item():.4f}",
            'dice': f"{dice_scores['overall']:.4f}"
        })

    epoch_loss = running_loss / len(dataloader)
    epoch_dice = np.mean(dice_scores_epoch)

    return epoch_loss, epoch_dice

def validate(model, dataloader, criterion, device):
    """Validate the model"""
    model.eval()

    running_loss = 0.0
    dice_scores_epoch = []
    class_dice_scores = defaultdict(list)

    pbar = tqdm(dataloader, desc='Validation')

    with torch.no_grad():
        for batch in pbar:
            images = batch['image'].to(device)
            masks = batch['mask'].to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, masks)

            # Calculate metrics
            dice_scores = calculate_dice_score(outputs, masks, CONFIG['num_classes'])
            dice_scores_epoch.append(dice_scores['overall'])

            for key, value in dice_scores.items():
                class_dice_scores[key].append(value)

            running_loss += loss.item()

            # Update progress bar
            pbar.set_postfix({
                'loss': f"{loss.item():.4f}",
                'dice': f"{dice_scores['overall']:.4f}"
            })

    epoch_loss = running_loss / len(dataloader)
    epoch_dice = np.mean(dice_scores_epoch)

    # Calculate mean per-class dice
    mean_class_dice = {k: np.mean(v) for k, v in class_dice_scores.items()}

    return epoch_loss, epoch_dice, mean_class_dice

print("✅ Training and validation functions defined")

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs, device, checkpoint_path, early_stopping_patience=5):
    """
    Main training loop with early stopping
    """
    history = {
        'train_loss': [],
        'train_dice': [],
        'val_loss': [],
        'val_dice': [],
        'val_dice_bg': [],
        'val_dice_artery': [],
        'val_dice_vein': [],
        'lr': []
    }

    best_val_dice = 0.0
    patience_counter = 0

    print(f"\n{'='*60}")
    print("TRAINING START")
    print(f"{'='*60}\n")

    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print(f"{'-'*60}")

        # Train
        train_loss, train_dice = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Validate
        val_loss, val_dice, class_dice = validate(
            model, val_loader, criterion, device
        )

        # Update learning rate
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        # Save history
        history['train_loss'].append(train_loss)
        history['train_dice'].append(train_dice)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        history['val_dice_bg'].append(class_dice['class_0'])
        history['val_dice_artery'].append(class_dice['class_1'])
        history['val_dice_vein'].append(class_dice['class_2'])
        history['lr'].append(current_lr)

        epoch_time = time.time() - epoch_start_time

        # Print epoch summary
        print(f"\n{'─'*60}")
        print(f"Epoch {epoch+1} Summary:")
        print(f"  Train Loss: {train_loss:.4f} | Train Dice: {train_dice:.4f}")
        print(f"  Val Loss:   {val_loss:.4f} | Val Dice:   {val_dice:.4f}")
        print(f"  Val Dice - BG: {class_dice['class_0']:.4f} | Artery: {class_dice['class_1']:.4f} | Vein: {class_dice['class_2']:.4f}")
        print(f"  Learning Rate: {current_lr:.6f}")
        print(f"  Time: {epoch_time:.2f}s")

        # Save best model
        if val_dice > best_val_dice:
            best_val_dice = val_dice
            patience_counter = 0

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_dice': val_dice,
                'val_loss': val_loss,
                'config': CONFIG
            }, checkpoint_path)

            print(f"  ✅ Best model saved! (Val Dice: {val_dice:.4f})")
        else:
            patience_counter += 1
            print(f"  ⏳ No improvement ({patience_counter}/{early_stopping_patience})")

        print(f"{'─'*60}")

        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"\n⚠️  Early stopping triggered after {epoch+1} epochs")
            print(f"Best validation Dice: {best_val_dice:.4f}")
            break

    print(f"\n{'='*60}")
    print("TRAINING COMPLETE")
    print(f"{'='*60}")
    print(f"Best Validation Dice: {best_val_dice:.4f}")

    return history

print("✅ Training loop defined")

In [ ]:
# Clear GPU cache
torch.cuda.empty_cache()

print(f"\n{'='*80}")
print(f"STARTING TRAINING: {CONFIG['model_name'].upper()}")
print(f"{'='*80}\n")

# Train the model
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=CONFIG['num_epochs'],
    device=device,
    checkpoint_path=CONFIG['checkpoint_path'],
    early_stopping_patience=CONFIG['early_stopping_patience']
)

print("\n✅ Training completed successfully!")

In [ ]:
# Save history to CSV
history_df = pd.DataFrame(history)
history_df['epoch'] = range(1, len(history_df) + 1)
history_df = history_df[['epoch', 'train_loss', 'train_dice', 'val_loss', 'val_dice',
                         'val_dice_bg', 'val_dice_artery', 'val_dice_vein', 'lr']]

history_df.to_csv(CONFIG['history_path'], index=False)
print(f"✅ Training history saved: {CONFIG['history_path']}")

# Display history
print("\nTraining History (last 10 epochs):")
print(history_df.tail(10).to_string(index=False))

In [ ]:
def plot_training_curves(history_df, output_dir):
    """Plot training and validation curves"""

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Loss curves
    axes[0, 0].plot(history_df['epoch'], history_df['train_loss'], label='Train Loss', marker='o')
    axes[0, 0].plot(history_df['epoch'], history_df['val_loss'], label='Val Loss', marker='o')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Loss Curves')
    axes[0, 0].legend()
    axes[0, 0].grid(alpha=0.3)

    # Dice curves
    axes[0, 1].plot(history_df['epoch'], history_df['train_dice'], label='Train Dice', marker='o')
    axes[0, 1].plot(history_df['epoch'], history_df['val_dice'], label='Val Dice', marker='o')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Dice Score')
    axes[0, 1].set_title('Dice Score Curves')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)

    # Per-class Dice
    axes[1, 0].plot(history_df['epoch'], history_df['val_dice_bg'], label='Background', marker='o')
    axes[1, 0].plot(history_df['epoch'], history_df['val_dice_artery'], label='Artery', marker='o')
    axes[1, 0].plot(history_df['epoch'], history_df['val_dice_vein'], label='Vein', marker='o')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Dice Score')
    axes[1, 0].set_title('Per-Class Dice Score')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

    # Learning rate
    axes[1, 1].plot(history_df['epoch'], history_df['lr'], marker='o', color='red')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_title('Learning Rate Schedule')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()

# Plot curves
plot_training_curves(history_df, CONFIG['output_dir'])
print("✅ Training curves saved")

In [ ]:
def visualize_predictions(model, dataset, device, num_samples=4):
    """Visualize model predictions on validation set"""

    model.eval()

    # Get random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    class_names = ['Background', 'Artery', 'Vein']
    colors = {0: [0, 0, 0], 1: [255, 0, 0], 2: [0, 0, 255]}

    with torch.no_grad():
        for idx, sample_idx in enumerate(indices):
            sample = dataset[sample_idx]

            image_tensor = sample['image'].unsqueeze(0).to(device)
            mask_true = sample['mask'].numpy()

            # Predict
            output = model(image_tensor)
            pred_mask = torch.argmax(output, dim=1).cpu().numpy()[0]

            # Denormalize image for display
            image_display = image_tensor.cpu().numpy()[0].transpose(1, 2, 0)
            image_display = image_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            image_display = np.clip(image_display, 0, 1)

            # Convert masks to RGB for visualization
            true_mask_rgb = np.zeros((mask_true.shape[0], mask_true.shape[1], 3), dtype=np.uint8)
            pred_mask_rgb = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3), dtype=np.uint8)

            for class_id, color in colors.items():
                true_mask_rgb[mask_true == class_id] = color
                pred_mask_rgb[pred_mask == class_id] = color

            # Plot
            axes[idx, 0].imshow(image_display)
            axes[idx, 0].set_title('Input Image')
            axes[idx, 0].axis('off')

            axes[idx, 1].imshow(true_mask_rgb)
            axes[idx, 1].set_title('Ground Truth')
            axes[idx, 1].axis('off')

            axes[idx, 2].imshow(pred_mask_rgb)
            axes[idx, 2].set_title('Prediction')
            axes[idx, 2].axis('off')

            # Overlay prediction on image
            overlay = image_display.copy()
            overlay_mask = pred_mask_rgb.astype(float) / 255.0
            overlay = (overlay * 0.6 + overlay_mask * 0.4)

            axes[idx, 3].imshow(overlay)
            axes[idx, 3].set_title('Overlay')
            axes[idx, 3].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['output_dir'], 'sample_predictions.png'), dpi=150, bbox_inches='tight')
    plt.show()

# Load best model and visualize
print("\nLoading best model for visualization...")
checkpoint = torch.load(CONFIG['checkpoint_path'], weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']} with Val Dice: {checkpoint['val_dice']:.4f}")

# Visualize predictions
print("\nGenerating sample predictions...")
visualize_predictions(model, val_dataset, device, num_samples=4)
print("✅ Sample predictions saved")

In [ ]:
# After training completes
print("\n✅ Training completed successfully!")

# Save training history to CSV
history_df = pd.DataFrame(history)
history_df.to_csv(CONFIG['history_path'], index=False)
print(f"✅ Training history saved to: {CONFIG['history_path']}")